In [1]:
import pandas as pd

# Carrega a base integrada de migração e emprego
indicadores_integrados = pd.read_parquet(
    "../data/processed/indicadores_integrados_municipais.parquet"
)

# Carrega os indicadores municipais de educação técnica
educacao_municipal = pd.read_parquet(
    "../data/processed/educacao/indicadores_educacao_municipal_2025.parquet"
)

# Integra as três dimensões pela chave municipal do IBGE
indicadores_completos = indicadores_integrados.merge(
    educacao_municipal.drop(columns=["municipio", "uf"], errors="ignore"),
    on="codigo_ibge",
    how="outer",
    validate="one_to_one",
    indicator="origem_integracao_educacao"
)

# Confere dimensões e cobertura entre as bases
print("Migração + emprego:", indicadores_integrados.shape)
print("Educação:", educacao_municipal.shape)
print("Base completa:", indicadores_completos.shape)

print("\nCobertura da integração:")
print(indicadores_completos["origem_integracao_educacao"].value_counts())

# Verifica se existem códigos IBGE duplicados
print("\nCódigos IBGE duplicados:",
      indicadores_completos["codigo_ibge"].duplicated().sum())

Migração + emprego: (1228, 12)
Educação: (3385, 10)
Base completa: (3689, 20)

Cobertura da integração:
origem_integracao_educacao
right_only    2461
both           924
left_only      304
Name: count, dtype: int64

Códigos IBGE duplicados: 0


In [2]:
# Carrega a referência oficial dos municípios
municipios_ibge = pd.read_parquet(
    "../data/processed/municipios_ibge.parquet"
)

# Adiciona os nomes e UFs de todos os municípios
indicadores_completos = indicadores_completos.drop(
    columns=["municipio", "uf"], errors="ignore"
).merge(
    municipios_ibge[["codigo_ibge", "municipio", "uf"]],
    on="codigo_ibge",
    how="left",
    validate="many_to_one"
)

# Verifica identificações geográficas ausentes
print("Municípios sem nome:", indicadores_completos["municipio"].isna().sum())
print("Municípios sem UF:", indicadores_completos["uf"].isna().sum())

# Verifica os valores ausentes em todas as colunas
display(
    indicadores_completos.isna().sum()
    .sort_values(ascending=False)
    .to_frame("valores_ausentes")
)

Municípios sem nome: 0
Municípios sem UF: 0


,valores_ausentes
variacao_%,2942
2024,2755
2025,2755
variacao_absoluta,2755
2026_parcial,2755
saldo,2635
admissoes,2635
desligamentos,2635
_merge,2461
percentual_ead,345


In [3]:
# Identifica os municípios presentes nas fontes originais
indicadores_completos["tem_migracao"] = (
    indicadores_completos["2025"].notna()
)

indicadores_completos["tem_emprego"] = (
    indicadores_completos["admissoes"].notna()
)

indicadores_completos["tem_educacao"] = (
    indicadores_completos["ofertas_cursos"].notna()
)

# Conta quantas dimensões possuem dados em cada município
indicadores_completos["fontes_disponiveis"] = (
    indicadores_completos[
        ["tem_migracao", "tem_emprego", "tem_educacao"]
    ].sum(axis=1)
)

# Resume a cobertura das três dimensões
display(
    indicadores_completos["fontes_disponiveis"]
    .value_counts()
    .sort_index()
    .to_frame("municipios")
)

# Identifica quantos municípios possuem dados nas três dimensões
print(
    "Municípios com três dimensões:",
    (indicadores_completos["fontes_disponiveis"] == 3).sum()
)


,municipios
fontes_disponiveis,
1,2628
2,438
3,623


Municípios com três dimensões: 623


In [4]:
# Agrupa municípios pelas combinações de fontes disponíveis
cobertura_fontes = (
    indicadores_completos.groupby(
        ["tem_migracao", "tem_emprego", "tem_educacao"]
    )
    .size()
    .reset_index(name="municipios")
)

# Ordena as combinações pelo número de municípios
cobertura_fontes = cobertura_fontes.sort_values(
    "municipios", ascending=False
)

# Exibe a distribuição
display(cobertura_fontes)

,tem_migracao,tem_emprego,tem_educacao,municipios
0,False,False,True,2461
6,True,True,True,623
2,False,True,True,180
5,True,True,False,137
4,True,False,True,121
1,False,True,False,114
3,True,False,False,53


In [5]:
# Seleciona municípios com registros de migração e emprego
municipios_priorizacao = indicadores_completos[
    indicadores_completos["tem_migracao"]
    & indicadores_completos["tem_emprego"]
].copy()

# Mantém informações territoriais e indicadores relevantes
colunas_priorizacao = [
    "codigo_ibge", "municipio", "uf",
    "2024", "2025", "variacao_absoluta", "variacao_%",
    "admissoes", "desligamentos", "saldo",
    "instituicoes_tecnicas", "cursos_distintos",
    "matriculas_tecnicas", "percentual_ead"
]

municipios_priorizacao = municipios_priorizacao[
    colunas_priorizacao
]

# Confere tamanho e cobertura educacional da base principal
print("Municípios elegíveis:", len(municipios_priorizacao))
print(
    "Com dados educacionais:",
    municipios_priorizacao["instituicoes_tecnicas"].notna().sum()
)

# Visualiza os primeiros registros
display(municipios_priorizacao.head())

Municípios elegíveis: 760
Com dados educacionais: 623


,codigo_ibge,municipio,uf,2024,2025,variacao_absoluta,variacao_%,admissoes,desligamentos,saldo,instituicoes_tecnicas,cursos_distintos,matriculas_tecnicas,percentual_ead
1,1100023,ARIQUEMES,RO,0.0,17.0,17.0,NaN,34.0,18.0,16.0,4.0,13.0,1185.0,30.23
2,1100049,CACOAL,RO,94.0,89.0,-5.0,-5.32,160.0,141.0,19.0,7.0,10.0,1389.0,34.09
3,1100056,CEREJEIRAS,RO,10.0,24.0,14.0,140.00,10.0,0.0,10.0,1.0,1.0,38.0,0.00
7,1100098,ESPIGÃO D'OESTE,RO,19.0,14.0,-5.0,-26.32,40.0,45.0,-5.0,NaN,NaN,NaN,NaN
8,1100106,GUAJARÁ-MIRIM,RO,143.0,130.0,-13.0,-9.09,26.0,16.0,10.0,1.0,5.0,534.0,0.00


In [6]:
# Carrega os indicadores municipais do ensino superior
superior_municipal = pd.read_parquet(
    "../data/processed/educacao/indicadores_ensino_superior_municipal_2024.parquet"
)

# Seleciona apenas a chave municipal e os indicadores educacionais
colunas_superior = [
    "codigo_ibge",
    "cursos_presenciais",
    "ies_presenciais",
    "matriculas_presenciais",
    "matriculas_estrangeiros_presencial",
    "matriculas_ead",
    "ingressantes_ead",
    "matriculas_estrangeiros_ead"
]

# Integra o ensino superior, preservando municípios de todas as fontes
indicadores_completos = indicadores_completos.merge(
    superior_municipal[colunas_superior],
    on="codigo_ibge",
    how="outer",
    validate="one_to_one",
    indicator="origem_integracao_superior"
)

# Verifica a integração
print("Dimensões:", indicadores_completos.shape)
print("\nCobertura:")
print(indicadores_completos["origem_integracao_superior"].value_counts())

print("\nCódigos duplicados:",
      indicadores_completos["codigo_ibge"].duplicated().sum())

print("Municípios sem nome:",
      indicadores_completos["municipio"].isna().sum())

Dimensões: (4441, 32)

Cobertura:
origem_integracao_superior
both          2799
left_only      890
right_only     752
Name: count, dtype: int64

Códigos duplicados: 0
Municípios sem nome: 752


In [7]:
# Carrega a referência oficial dos municípios
municipios_ibge = pd.read_parquet(
    "../data/processed/municipios_ibge.parquet"
)

# Adiciona os nomes e UFs oficiais como colunas auxiliares
indicadores_completos = indicadores_completos.merge(
    municipios_ibge[["codigo_ibge", "municipio", "uf"]],
    on="codigo_ibge",
    how="left",
    suffixes=("", "_ibge"),
    validate="one_to_one"
)

# Preenche apenas os nomes e UFs que estavam ausentes
indicadores_completos["municipio"] = (
    indicadores_completos["municipio"]
    .fillna(indicadores_completos["municipio_ibge"])
)

indicadores_completos["uf"] = (
    indicadores_completos["uf"]
    .fillna(indicadores_completos["uf_ibge"])
)

# Remove as colunas auxiliares
indicadores_completos = indicadores_completos.drop(
    columns=["municipio_ibge", "uf_ibge"]
)

# Valida a identificação territorial
print("Municípios:", len(indicadores_completos))
print("Sem nome:", indicadores_completos["municipio"].isna().sum())
print("Sem UF:", indicadores_completos["uf"].isna().sum())
print("Códigos duplicados:",
      indicadores_completos["codigo_ibge"].duplicated().sum())

Municípios: 4441
Sem nome: 0
Sem UF: 0
Códigos duplicados: 0


In [8]:
# Atualiza os indicadores de presença das fontes
indicadores_completos["tem_migracao"] = indicadores_completos["2025"].notna()
indicadores_completos["tem_emprego"] = indicadores_completos["admissoes"].notna()
indicadores_completos["tem_educacao"] = indicadores_completos["ofertas_cursos"].notna()

# Identifica municípios com registros de ensino superior presencial ou EAD
indicadores_completos["tem_superior"] = indicadores_completos[
    ["matriculas_presenciais", "matriculas_ead"]
].notna().any(axis=1)

# Recalcula a quantidade de dimensões disponíveis
colunas_cobertura = [
    "tem_migracao",
    "tem_emprego",
    "tem_educacao",
    "tem_superior"
]

indicadores_completos["fontes_disponiveis"] = (
    indicadores_completos[colunas_cobertura].sum(axis=1)
)

# Exibe a distribuição de cobertura
display(
    indicadores_completos["fontes_disponiveis"]
    .value_counts()
    .sort_index()
    .to_frame("municipios")
)

# Confere a cobertura individual de cada fonte
display(
    indicadores_completos[colunas_cobertura]
    .sum()
    .to_frame("municipios")
)

,municipios
fontes_disponiveis,
1,1536
2,1919
3,394
4,592


,municipios
tem_migracao,934
tem_emprego,1054
tem_educacao,3385
tem_superior,3551


In [9]:
# Remove colunas auxiliares utilizadas durante as integrações
indicadores_completos = indicadores_completos.drop(
    columns=[
        "_merge",
        "origem_integracao_educacao",
        "origem_integracao_superior"
    ],
    errors="ignore"
)

# Confere os totais e a integridade da base
print("Municípios:", len(indicadores_completos))
print("Códigos duplicados:",
      indicadores_completos["codigo_ibge"].duplicated().sum())

print("Registros migratórios 2025:",
      indicadores_completos["2025"].sum())

print("Admissões CAGED 2025:",
      indicadores_completos["admissoes"].sum())

print("Matrículas técnicas 2025:",
      indicadores_completos["matriculas_tecnicas"].sum())

print("Matrículas superiores presenciais 2024:",
      indicadores_completos["matriculas_presenciais"].sum())

print("Matrículas superiores EAD 2024:",
      indicadores_completos["matriculas_ead"].sum())

Municípios: 4441
Códigos duplicados: 0
Registros migratórios 2025: 236777.0
Admissões CAGED 2025: 393415.0
Matrículas técnicas 2025: 2490145.0
Matrículas superiores presenciais 2024: 5037875.0
Matrículas superiores EAD 2024: 5186811.0


In [10]:
# Define o caminho da base municipal consolidada
arquivo_saida = "../data/processed/indicadores_completos_municipais.parquet"

# Salva os indicadores integrados sem incluir o índice do DataFrame
indicadores_completos.to_parquet(
    arquivo_saida,
    index=False
)

# Confirma a exportação e suas dimensões
print("Arquivo salvo:", arquivo_saida)
print("Dimensões:", indicadores_completos.shape)

Arquivo salvo: ../data/processed/indicadores_completos_municipais.parquet
Dimensões: (4441, 30)


### Construindo o ranking territorial

In [11]:
# Seleciona municípios presentes nas duas fontes principais
municipios_ranking = indicadores_completos[
    indicadores_completos["tem_migracao"]
    & indicadores_completos["tem_emprego"]
].copy()

# Examina a distribuição dos indicadores candidatos ao ranking
display(
    municipios_ranking[["2025", "admissoes"]]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
)

# Confere o número de municípios analisados
print("Municípios elegíveis:", len(municipios_ranking))

,2025,admissoes
count,760.000000,760.000000
mean,306.332895,501.839474
std,2020.452369,2094.218759
min,0.000000,0.000000
25%,16.000000,33.000000
50%,46.500000,90.000000
75%,135.000000,287.750000
90%,413.700000,947.100000
95%,810.700000,1807.950000
99%,3393.470000,6356.570000


Municípios elegíveis: 760


In [15]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Reduz a assimetria dos indicadores com transformação logarítmica
municipios_ranking["log_migracao"] = np.log1p(
    municipios_ranking["2025"]
)

municipios_ranking["log_admissoes"] = np.log1p(
    municipios_ranking["admissoes"]
)

# Normaliza os indicadores transformados para a escala de 0 a 1
scaler = MinMaxScaler()

municipios_ranking[["score_migracao", "score_emprego"]] = (
    scaler.fit_transform(
        municipios_ranking[["log_migracao", "log_admissoes"]]
    )
)

# Confere os intervalos dos indicadores normalizados
display(
    municipios_ranking[["score_migracao", "score_emprego"]]
    .describe()
)

,score_migracao,score_emprego
count,760.000000,760.000000
mean,0.361011,0.445746
std,0.175391,0.155363
min,0.000000,0.000000
25%,0.268321,0.337052
50%,0.365628,0.431151
75%,0.465256,0.541513
max,1.000000,1.000000


In [16]:
# Define cenários alternativos de ponderação
cenarios = {
    "equilibrado": (0.5, 0.5),
    "prioridade_migracao": (0.6, 0.4),
    "prioridade_emprego": (0.4, 0.6)
}

# Calcula a pontuação de cada município nos três cenários
for nome, (peso_migracao, peso_emprego) in cenarios.items():
    municipios_ranking[f"score_{nome}"] = (
        municipios_ranking["score_migracao"] * peso_migracao
        + municipios_ranking["score_emprego"] * peso_emprego
    )

# Calcula a posição de cada município em cada cenário
for nome in cenarios:
    municipios_ranking[f"rank_{nome}"] = (
        municipios_ranking[f"score_{nome}"]
        .rank(ascending=False, method="min")
        .astype(int)
    )

# Exibe os 15 municípios mais bem posicionados no cenário equilibrado
display(
    municipios_ranking.sort_values("rank_equilibrado")[
        [
            "municipio", "uf", "2025", "admissoes",
            "score_equilibrado",
            "rank_equilibrado",
            "rank_prioridade_migracao",
            "rank_prioridade_emprego"
        ]
    ].head(15)
)

,municipio,uf,2025,admissoes,score_equilibrado,rank_equilibrado,rank_prioridade_migracao,rank_prioridade_emprego
3166,SÃO PAULO,SP,38523.0,34973.0,1.000000,1,1,1
3316,CURITIBA,PR,12861.0,34731.0,0.947722,2,2,2
123,BOA VISTA,RR,35014.0,7464.0,0.921671,3,3,3
3608,CHAPECÓ,SC,4399.0,12807.0,0.849253,4,5,4
96,MANAUS,AM,8759.0,5587.0,0.842220,5,4,8
3624,FLORIANÓPOLIS,SC,4508.0,8758.0,0.832252,6,6,6
3669,JOINVILLE,SC,3492.0,10862.0,0.830450,7,8,5
3293,CASCAVEL,PR,3143.0,11027.0,0.826186,8,9,7
3869,CAXIAS DO SUL,RS,2709.0,10164.0,0.815258,9,10,9
2675,RIO DE JANEIRO,RJ,7426.0,3573.0,0.813044,10,7,10


In [17]:
# Calcula quanto a posição de cada município muda entre os cenários
municipios_ranking["variacao_ranking"] = (
    municipios_ranking["rank_prioridade_migracao"]
    - municipios_ranking["rank_prioridade_emprego"]
).abs()

# Resume a magnitude das mudanças de posição
print("Variação mediana:",
      municipios_ranking["variacao_ranking"].median())

print("Variação média:",
      municipios_ranking["variacao_ranking"].mean().round(2))

print("Maior variação:",
      municipios_ranking["variacao_ranking"].max())

# Mede a correlação entre as posições nos dois cenários extremos
correlacao = municipios_ranking[
    ["rank_prioridade_migracao", "rank_prioridade_emprego"]
].corr(method="spearman")

display(correlacao)

# Compara os municípios presentes no Top 20 dos três cenários
tops = {
    nome: set(municipios_ranking.nsmallest(
        20, f"rank_{nome}"
    )["codigo_ibge"])
    for nome in cenarios
}

print("Municípios comuns aos três Top 20:",
      len(set.intersection(*tops.values())))

Variação mediana: 14.0
Variação média: 23.97
Maior variação: 214


,rank_prioridade_migracao,rank_prioridade_emprego
rank_prioridade_migracao,1.000,0.986
rank_prioridade_emprego,0.986,1.000


Municípios comuns aos três Top 20: 18


### Investigar os municípios mais sensíveis
Antes de consolidar a metodologia, vamos identificar quais municípios apresentam as maiores diferenças de posição e entender por quê.

In [18]:
# Seleciona os municípios com maiores mudanças entre os cenários
municipios_sensiveis = municipios_ranking.nlargest(
    15, "variacao_ranking"
)

# Compara os indicadores originais e as posições nos dois cenários
display(
    municipios_sensiveis[
        [
            "municipio", "uf",
            "2025", "admissoes",
            "rank_prioridade_migracao",
            "rank_prioridade_emprego",
            "variacao_ranking"
        ]
    ]
)

,municipio,uf,2025,admissoes,rank_prioridade_migracao,rank_prioridade_emprego,variacao_ranking
4005,PINTO BANDEIRA,RS,0.0,586.0,667,453,214
3780,TREZE TÍLIAS,SC,0.0,509.0,672,467,205
3697,NOVA VENEZA,SC,0.0,426.0,676,494,182
3336,FRANCISCO ALVES,PR,329.0,9.0,314,486,172
3940,IPÊ,RS,0.0,323.0,679,529,150
2905,IRACEMÁPOLIS,SP,126.0,10.0,424,552,128
935,PARNAMIRIM,RN,121.0,10.0,430,557,127
3980,NOVA PÁDUA,RS,0.0,245.0,680,563,117
3914,FLORES DA CUNHA,RS,16.0,1807.0,257,144,113
2854,FRANCISCO MORATO,SP,96.0,11.0,450,562,112


In [19]:
# Cria uma cópia do ranking para a apresentação
ranking_municipal_2025 = municipios_ranking.copy()

# Converte a pontuação equilibrada para a escala de 0 a 100
ranking_municipal_2025["indice_oportunidade"] = (
    ranking_municipal_2025["score_equilibrado"] * 100
).round(2)

# Organiza os municípios pela posição no cenário equilibrado
ranking_municipal_2025 = ranking_municipal_2025.sort_values(
    "rank_equilibrado"
)

# Seleciona os indicadores essenciais para o ranking
colunas_ranking = [
    "codigo_ibge", "municipio", "uf",
    "2025", "admissoes", "saldo",
    "score_migracao", "score_emprego",
    "indice_oportunidade", "rank_equilibrado",
    "rank_prioridade_migracao", "rank_prioridade_emprego",
    "variacao_ranking"
]

ranking_municipal_2025 = ranking_municipal_2025[colunas_ranking]

# Exibe os dez primeiros municípios
display(ranking_municipal_2025.head(10))

,codigo_ibge,municipio,uf,2025,admissoes,saldo,score_migracao,score_emprego,indice_oportunidade,rank_equilibrado,rank_prioridade_migracao,rank_prioridade_emprego,variacao_ranking
3166,3550308.0,SÃO PAULO,SP,38523.0,34973.0,6489.0,1.000000,1.000000,100.00,1,1,1,0
3316,4106902.0,CURITIBA,PR,12861.0,34731.0,7422.0,0.896108,0.999336,94.77,2,2,2,0
123,1400100.0,BOA VISTA,RR,35014.0,7464.0,616.0,0.990955,0.852387,92.17,3,3,3,0
3608,4204202.0,CHAPECÓ,SC,4399.0,12807.0,2416.0,0.794519,0.903986,84.93,4,5,4,1
96,1302603.0,MANAUS,AM,8759.0,5587.0,501.0,0.859733,0.824707,84.22,5,4,8,4
3624,4205407.0,FLORIANÓPOLIS,SC,4508.0,8758.0,1509.0,0.796837,0.867666,83.23,6,6,6,0
3669,4209102.0,JOINVILLE,SC,3492.0,10862.0,1842.0,0.772657,0.888243,83.05,7,8,5,3
3293,4104808.0,CASCAVEL,PR,3143.0,11027.0,1939.0,0.762688,0.889684,82.62,8,9,7,2
3869,4305108.0,CAXIAS DO SUL,RS,2709.0,10164.0,1416.0,0.748620,0.881895,81.53,9,10,9,1
2675,3304557.0,RIO DE JANEIRO,RJ,7426.0,3573.0,248.0,0.844099,0.781988,81.30,10,7,10,3


In [20]:
# Calcula a diferença entre os componentes normalizados do índice
ranking_municipal_2025["diferenca_componentes"] = (
    ranking_municipal_2025["score_migracao"]
    - ranking_municipal_2025["score_emprego"]
)

# Identifica municípios relativamente mais destacados em migração
print("Maior destaque migratório:")
display(
    ranking_municipal_2025.nlargest(10, "diferenca_componentes")[
        ["municipio", "uf", "2025", "admissoes", "diferenca_componentes"]
    ]
)

# Identifica municípios relativamente mais destacados em admissões
print("Maior destaque em emprego:")
display(
    ranking_municipal_2025.nsmallest(10, "diferenca_componentes")[
        ["municipio", "uf", "2025", "admissoes", "diferenca_componentes"]
    ]
)

Maior destaque migratório:


,municipio,uf,2025,admissoes,diferenca_componentes
2528,VIÇOSA,MG,77.0,0.0,0.412605
3336,FRANCISCO ALVES,PR,329.0,9.0,0.329124
834,JIJOCA DE JERICOACOARA,CE,21.0,0.0,0.292739
4396,PALMINÓPOLIS,GO,16.0,0.0,0.268321
3583,BELMONTE,SC,14.0,0.0,0.256468
124,BONFIM,RR,634.0,41.0,0.253945
131,PACARAIMA,RR,2169.0,142.0,0.253222
3255,ATALAIA,PR,11.0,0.0,0.235335
2861,GUAIÇARA,SP,11.0,0.0,0.235335
2905,IRACEMÁPOLIS,SP,126.0,10.0,0.229579


Maior destaque em emprego:


,municipio,uf,2025,admissoes,diferenca_componentes
4005,PINTO BANDEIRA,RS,0.0,586.0,-0.609330
3780,TREZE TÍLIAS,SC,0.0,509.0,-0.595890
3697,NOVA VENEZA,SC,0.0,426.0,-0.578912
3940,IPÊ,RS,0.0,323.0,-0.552528
3980,NOVA PÁDUA,RS,0.0,245.0,-0.526204
3301,CIDADE GAÚCHA,PR,0.0,206.0,-0.509705
3719,POMERODE,SC,0.0,153.0,-0.481436
3761,SÃO JOSÉ DO CEDRO,SC,0.0,147.0,-0.477637
3914,FLORES DA CUNHA,RS,16.0,1807.0,-0.448532
1658,LUÍS EDUARDO MAGALHÃES,BA,0.0,90.0,-0.431151


## Carregando arquivos com dados sobre população geral Brasil (IBGE)

In [45]:
# Define o caminho do arquivo Excel
arquivo_populacao = "../data/raw/populacao/POP2026_20261029.xlsx"

# Identifica as planilhas disponíveis
excel = pd.ExcelFile(arquivo_populacao)
print("Planilhas:", excel.sheet_names)

# Carrega a segunda planilha, com população municipal
populacao_municipal = pd.read_excel(
    arquivo_populacao,
    sheet_name=1,
    dtype={"COD. MUNIC": str}

)

# Remove colunas completamente vazias
populacao_municipal = populacao_municipal.dropna(
    axis=1,
    how="all"
).drop('Unnamed: 5', axis =1)



# Examina a estrutura e os primeiros registros
print("Dimensões:", populacao_municipal.shape)
print("Colunas:", populacao_municipal.columns.tolist())

display(populacao_municipal.head())

Planilhas: ['BRASIL E UFS', 'MUNICÍPIOS']
Dimensões: (5571, 5)
Colunas: ['UF', 'COD. UF', 'COD. MUNIC', 'NOME DO MUNICÍPIO', 'POPULAÇÃO ESTIMADA']


,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,POPULAÇÃO ESTIMADA
0,RO,11,00015,Alta Floresta D'Oeste,22724
1,RO,11,00023,Ariquemes,109731
2,RO,11,00031,Cabixi,5640
3,RO,11,00049,Cacoal,98884
4,RO,11,00056,Cerejeiras,16957


In [46]:
# Concatena os códigos da UF e do município
populacao_municipal["codigo_ibge"] = (
    populacao_municipal["COD. UF"].astype(str).str.zfill(2)
    + populacao_municipal["COD. MUNIC"].astype(str).str.zfill(5)
).astype("int64")

# Renomeia a coluna de população para facilitar a integração
populacao_municipal = populacao_municipal.rename(
    columns={"POPULAÇÃO ESTIMADA": "populacao_2026"}
)

# Verifica a integridade dos dados
print("Municípios:", len(populacao_municipal))
print("Códigos duplicados:", populacao_municipal["codigo_ibge"].duplicated().sum())
print("Populações ausentes:", populacao_municipal["populacao_2026"].isna().sum())

# Confere a estrutura resultante
display(
    populacao_municipal[
        ["codigo_ibge", "NOME DO MUNICÍPIO", "UF", "populacao_2026"]
    ].head()
)

Municípios: 5571
Códigos duplicados: 0
Populações ausentes: 0


,codigo_ibge,NOME DO MUNICÍPIO,UF,populacao_2026
0,1100015,Alta Floresta D'Oeste,RO,22724
1,1100023,Ariquemes,RO,109731
2,1100031,Cabixi,RO,5640
3,1100049,Cacoal,RO,98884
4,1100056,Cerejeiras,RO,16957


### Participação de imigrantes nas admissões

In [47]:
# Carrega os indicadores municipais do CAGED geral
emprego_geral = pd.read_parquet(
    "../data/processed/trabalho/emprego_geral_municipal_2025.parquet"
)

# Renomeia as variáveis para diferenciá-las dos indicadores de migrantes
emprego_geral = emprego_geral.rename(columns={
    "admissoes": "admissoes_gerais",
    "desligamentos": "desligamentos_gerais",
    "saldo": "saldo_geral"
})

# Integra o mercado geral à base consolidada, preservando todos os municípios
indicadores_completos = indicadores_completos.merge(
    emprego_geral[
        ["codigo_ibge", "admissoes_gerais", "desligamentos_gerais", "saldo_geral"]
    ],
    on="codigo_ibge",
    how="outer",
    validate="one_to_one",
    indicator="origem_emprego_geral"
)

# Verifica as dimensões e a cobertura da integração
print("Dimensões:", indicadores_completos.shape)
print("Códigos duplicados:", indicadores_completos["codigo_ibge"].duplicated().sum())

display(indicadores_completos["origem_emprego_geral"].value_counts())

# Confere quantos municípios possuem os dois indicadores de admissões
print(
    "Municípios com dados de emprego geral e migrante:",
    (
        indicadores_completos["admissoes_gerais"].notna()
        & indicadores_completos["admissoes"].notna()
    ).sum()
)

Dimensões: (5567, 34)
Códigos duplicados: 0


origem_emprego_geral
both          4440
right_only    1126
left_only        1
Name: count, dtype: int64

Municípios com dados de emprego geral e migrante: 1054


In [50]:
# Seleciona municípios com dados de emprego geral e migrante
comparacao_emprego = indicadores_completos[
    indicadores_completos["admissoes"].notna()
    & indicadores_completos["admissoes_gerais"].notna()
].copy()

# Calcula a diferença entre admissões gerais e de migrantes
comparacao_emprego["diferenca_admissoes"] = (
    comparacao_emprego["admissoes_gerais"]
    - comparacao_emprego["admissoes"]
)

# Identifica municípios com mais admissões migrantes do que gerais
inconsistencias = comparacao_emprego[
    comparacao_emprego["diferenca_admissoes"] < 0
]

print("Municípios comparados:", len(comparacao_emprego))
print("Municípios com inconsistências:", len(inconsistencias))

# Exibe eventuais inconsistências
display(
    inconsistencias[
        ["municipio", "uf", "admissoes", "admissoes_gerais",
         "diferenca_admissoes"]
    ]
)

Municípios comparados: 1054
Municípios com inconsistências: 0


,municipio,uf,admissoes,admissoes_gerais,diferenca_admissoes


In [51]:
import numpy as np

# Calcula a participação migrante apenas quando há admissões gerais
comparacao_emprego["participacao_migrante_pct"] = np.where(
    comparacao_emprego["admissoes_gerais"] > 0,
    comparacao_emprego["admissoes"]
    / comparacao_emprego["admissoes_gerais"] * 100,
    np.nan
)

# Verifica a distribuição e possíveis percentuais inválidos
print("Percentuais ausentes:",
      comparacao_emprego["participacao_migrante_pct"].isna().sum())

print("Percentuais acima de 100%:",
      (comparacao_emprego["participacao_migrante_pct"] > 100).sum())

display(
    comparacao_emprego["participacao_migrante_pct"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
)

# Examina os municípios com maior participação e pelo menos 100 admissões gerais
display(
    comparacao_emprego[
        comparacao_emprego["admissoes_gerais"] >= 100
    ]
    .nlargest(15, "participacao_migrante_pct")[
        ["municipio", "uf", "admissoes",
         "admissoes_gerais", "participacao_migrante_pct"]
    ]
)

Percentuais ausentes: 0
Percentuais acima de 100%: 0


count    1054.000000
mean        3.876746
std         6.446814
min         0.000000
25%         0.404833
50%         1.348145
75%         4.836849
90%         9.756327
95%        15.286325
99%        32.712808
max        54.826255
Name: participacao_migrante_pct, dtype: float64

,municipio,uf,admissoes,admissoes_gerais,participacao_migrante_pct
146,PACARAIMA,RR,142.0,259.0,54.826255
4410,GUATAMBÚ,SC,1826.0,3460.0,52.774566
4708,CHUÍ,RS,782.0,1493.0,52.377763
4919,PINTO BANDEIRA,RS,586.0,1144.0,51.223776
4683,CAPÃO BONITO DO SUL,RS,86.0,176.0,48.863636
4859,MONTE BELO DO SUL,RS,380.0,865.0,43.930636
4720,CORONEL PILAR,RS,81.0,192.0,42.187500
4880,NOVA PÁDUA,RS,245.0,594.0,41.245791
4480,NOVO HORIZONTE,SC,81.0,198.0,40.909091
4972,SANTA TEREZA,RS,67.0,179.0,37.430168


In [52]:
# Calcula os totais apenas nos 1.054 municípios comparáveis
total_migrantes = comparacao_emprego["admissoes"].sum()
total_geral = comparacao_emprego["admissoes_gerais"].sum()

# Calcula a participação ponderada pelo volume de admissões
participacao_ponderada = total_migrantes / total_geral * 100

print(f"Admissões migrantes: {total_migrantes:,.0f}")
print(f"Admissões gerais nos mesmos municípios: {total_geral:,.0f}")
print(f"Participação ponderada: {participacao_ponderada:.2f}%")
print(
    f"Média simples dos municípios: "
    f"{comparacao_emprego['participacao_migrante_pct'].mean():.2f}%"
)

Admissões migrantes: 393,415
Admissões gerais nos mesmos municípios: 22,215,954
Participação ponderada: 1.77%
Média simples dos municípios: 3.88%


**Interpretação**: os municípios com maior participação de migrantes nas admissões tendem a ter menor peso no volume total de contratações. Por isso, a média simples é superior à participação ponderada.

**Uma ressalva**: 1,77% é a participação nos 1.054 municípios comparados, não no Brasil inteiro. Além disso, precisamos verificar a compatibilidade dos recortes do CAGED antes de considerar essa participação definitiva.

In [53]:
# Integra a população à base consolidada de municípios
indicadores_completos = indicadores_completos.merge(
    populacao_municipal[["codigo_ibge", "populacao_2026"]],
    on="codigo_ibge",
    how="left",
    validate="one_to_one"
)

# Calcula registros migratórios por 10 mil habitantes
indicadores_completos["registros_por_10mil_hab"] = np.where(
    indicadores_completos["populacao_2026"] > 0,
    indicadores_completos["2025"]
    / indicadores_completos["populacao_2026"] * 10000,
    np.nan
)

# Verifica cobertura e distribuição do indicador
print("Municípios:", len(indicadores_completos))
print("Populações ausentes:",
      indicadores_completos["populacao_2026"].isna().sum())

display(
    indicadores_completos["registros_por_10mil_hab"]
    .describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99])
)

Municípios: 5567
Populações ausentes: 0


count    934.000000
mean      24.501115
std       54.964521
min        0.000000
50%        8.165078
75%       27.125715
90%       57.823798
95%       88.140501
99%      229.703050
max      898.806564
Name: registros_por_10mil_hab, dtype: float64

Temos agora três perspectivas para analisar as oportunidades territoriais da UNIESF:
1. Volume: registros migratórios e admissões de migrantes.
2. Participação no emprego: percentual de admissões de migrantes no CAGED geral.
3. Intensidade populacional: registros migratórios por 10 mil habitantes.

In [59]:
# Seleciona municípios com indicadores migratórios disponíveis
intensidade_migratoria = indicadores_completos[
    indicadores_completos["registros_por_10mil_hab"].notna()
].copy()

# Ordena os municípios pela intensidade dos registros migratórios
ranking_intensidade = intensidade_migratoria.sort_values(
    "registros_por_10mil_hab",
    ascending=False
)

# Exibe os 20 municípios com maior intensidade
display(
    ranking_intensidade[
        [
            "municipio",
            "uf",
            "2025",
            "populacao_2026",
            "registros_por_10mil_hab",
            "admissoes",
            "admissoes_gerais"
        ]
    ].head(20)
)

,municipio,uf,2025,populacao_2026,registros_por_10mil_hab,admissoes,admissoes_gerais
146,PACARAIMA,RR,2169.0,24132,898.806564,142.0,259.0
138,BOA VISTA,RR,35014.0,500965,698.931063,7464.0,44363.0
4708,CHUÍ,RS,273.0,6409,425.963489,782.0,1493.0
4638,BARRA DO QUARAÍ,RS,174.0,4340,400.921659,42.0,643.0
139,BONFIM,RR,634.0,15922,398.191182,41.0,518.0
4027,FRANCISCO ALVES,PR,329.0,8677,379.163305,9.0,497.0
3175,ARMAÇÃO DOS BÚZIOS,RJ,1124.0,44365,253.352868,965.0,9243.0
42,PIMENTEIRAS DO OESTE,RO,55.0,2308,238.301560,NaN,216.0
4386,DIONÍSIO CERQUEIRA,SC,359.0,15337,234.074460,240.0,1559.0
4035,GUAÍRA,PR,773.0,33145,233.217680,130.0,2876.0


In [60]:
# Seleciona os indicadores complementares da base consolidada
indicadores_relativos = indicadores_completos[
    ["codigo_ibge", "registros_por_10mil_hab",
     "admissoes", "admissoes_gerais"]
].copy()

# Integra os indicadores ao ranking de volume
comparacao_rankings = ranking_municipal_2025.merge(
    indicadores_relativos.drop(columns=["admissoes"]),
    on="codigo_ibge",
    how="left",
    validate="one_to_one"
)

# Calcula a participação das admissões migrantes no emprego geral
comparacao_rankings["participacao_migrante_pct"] = (
    comparacao_rankings["admissoes"]
    / comparacao_rankings["admissoes_gerais"] * 100
)

# Verifica a distribuição dos três indicadores
display(
    comparacao_rankings[
        ["indice_oportunidade", "registros_por_10mil_hab",
         "participacao_migrante_pct"]
    ].describe()
)

# Examina as correlações entre as perspectivas
display(
    comparacao_rankings[
        ["indice_oportunidade", "registros_por_10mil_hab",
         "participacao_migrante_pct"]
    ].corr(method="spearman")
)

,indice_oportunidade,registros_por_10mil_hab,participacao_migrante_pct
count,760.000000,760.000000,760.000000
mean,40.337921,26.554486,4.119072
std,15.226074,58.738090,6.323974
min,0.000000,0.000000,0.000000
25%,30.687500,2.877221,0.472626
50%,38.920000,9.849563,1.598387
75%,49.317500,29.240585,5.487143
max,100.000000,898.806564,54.826255


,indice_oportunidade,registros_por_10mil_hab,participacao_migrante_pct
indice_oportunidade,1.000000,0.563674,0.347025
registros_por_10mil_hab,0.563674,1.000000,0.707085
participacao_migrante_pct,0.347025,0.707085,1.000000


### Implicações para o projeto
Esses resultados sugerem dois perfis de oportunidades:

- Grandes polos de alcance: municípios com altos volumes absolutos de registros migratórios e contratações, como São Paulo e Curitiba.
- Territórios de maior intensidade: municípios onde os registros migratórios e as admissões de migrantes têm maior peso relativo, como Pacaraima e Chuí.

**Alguns municípios podem combinar os dois perfis.**

Para a UNIESF, essa distinção é útil porque municípios com grande volume podem representar maior alcance potencial, enquanto municípios de maior intensidade podem indicar contextos nos quais ações específicas para migrantes merecem investigação.

#### Normalizar os indicadores de intensidade
Assim como fizemos no ranking de volume, vamos usar log1p para reduzir a influência dos valores extremos e Min-Max para colocar as variáveis na mesma escala.

In [61]:
# Cria uma cópia para construir o ranking de intensidade
ranking_intensidade_2025 = comparacao_rankings.copy()

# Aplica transformação logarítmica aos indicadores relativos
ranking_intensidade_2025["log_intensidade_migratoria"] = np.log1p(
    ranking_intensidade_2025["registros_por_10mil_hab"]
)

ranking_intensidade_2025["log_participacao_emprego"] = np.log1p(
    ranking_intensidade_2025["participacao_migrante_pct"]
)

# Normaliza os dois indicadores para a escala de 0 a 1
scaler_intensidade = MinMaxScaler()

ranking_intensidade_2025[
    ["score_intensidade_migratoria", "score_participacao_emprego"]
] = scaler_intensidade.fit_transform(
    ranking_intensidade_2025[
        ["log_intensidade_migratoria", "log_participacao_emprego"]
    ]
)

# Calcula o índice exploratório com pesos iguais
ranking_intensidade_2025["indice_intensidade"] = (
    0.5 * ranking_intensidade_2025["score_intensidade_migratoria"]
    + 0.5 * ranking_intensidade_2025["score_participacao_emprego"]
) * 100

# Exibe os 15 municípios com maior intensidade combinada
display(
    ranking_intensidade_2025.nlargest(15, "indice_intensidade")[
        [
            "municipio", "uf",
            "registros_por_10mil_hab",
            "participacao_migrante_pct",
            "indice_intensidade",
            "indice_oportunidade"
        ]
    ]
)

,municipio,uf,registros_por_10mil_hab,participacao_migrante_pct,indice_intensidade,indice_oportunidade
77,PACARAIMA,RR,898.806564,54.826255,100.000000,60.10
92,CHUÍ,RS,425.963489,52.377763,93.962751,58.42
94,GUATAMBÚ,SC,108.575450,52.774566,84.057398,58.11
2,BOA VISTA,RR,698.931063,16.824832,83.961822,92.17
80,ITAPIRANGA,SC,222.567288,23.289391,79.419522,59.82
74,SEARA,SC,195.079713,23.996656,78.811986,60.48
209,ROCHEDO,MS,180.196578,22.714681,77.577283,48.14
211,NOVA ARAÇÁ,RS,137.308749,26.445644,77.408247,48.10
66,CAFELÂNDIA,PR,170.496301,19.574845,75.407352,61.22
130,DIONÍSIO CERQUEIRA,SC,234.074460,15.394484,74.901937,54.08


#### A análise permite distinguir três perfis preliminares:
- Polos de alcance: municípios com elevado volume absoluto, como São Paulo e Curitiba.
- Polos de intensidade: municípios com participação migrante expressiva em relação à população ou às contratações locais, como Pacaraima e Chuí.
- Polos mistos: municípios que se destacam nas duas perspectivas, como Boa Vista e Chapecó.

In [62]:
# Calcula a posição dos municípios no ranking de intensidade
ranking_intensidade_2025["rank_intensidade"] = (
    ranking_intensidade_2025["indice_intensidade"]
    .rank(ascending=False, method="min")
    .astype(int)
)

# Compara a posição de volume e intensidade
ranking_intensidade_2025["diferenca_ranks"] = (
    ranking_intensidade_2025["rank_equilibrado"]
    - ranking_intensidade_2025["rank_intensidade"]
)

# Calcula a correlação de Spearman entre os rankings
correlacao = ranking_intensidade_2025[
    ["rank_equilibrado", "rank_intensidade"]
].corr(method="spearman").iloc[0, 1]

print(f"Correlação entre rankings: {correlacao:.3f}")

# Identifica municípios presentes nos dois Top 20
top_volume = set(
    ranking_intensidade_2025.nsmallest(20, "rank_equilibrado")["codigo_ibge"]
)

top_intensidade = set(
    ranking_intensidade_2025.nsmallest(20, "rank_intensidade")["codigo_ibge"]
)

print("Municípios comuns aos dois Top 20:",
      len(top_volume & top_intensidade))

# Exibe os municípios presentes nos dois grupos
display(
    ranking_intensidade_2025[
        ranking_intensidade_2025["codigo_ibge"].isin(top_volume & top_intensidade)
    ][
        ["municipio", "uf", "rank_equilibrado",
         "rank_intensidade", "indice_oportunidade",
         "indice_intensidade"]
    ].sort_values("rank_equilibrado")
)

Correlação entre rankings: 0.502
Municípios comuns aos dois Top 20: 2


,municipio,uf,rank_equilibrado,rank_intensidade,indice_oportunidade,indice_intensidade
2,BOA VISTA,RR,3,4,92.17,83.961822
3,CHAPECÓ,SC,4,11,84.93,73.864322


In [63]:
# Define cenários alternativos para os dois indicadores de intensidade
pesos_intensidade = {
    "equilibrado": (0.5, 0.5),
    "migracao": (0.6, 0.4),
    "emprego": (0.4, 0.6)
}

# Calcula os índices e rankings nos três cenários
for nome, (peso_migracao, peso_emprego) in pesos_intensidade.items():
    coluna_indice = f"indice_intensidade_{nome}"
    coluna_rank = f"rank_intensidade_{nome}"

    ranking_intensidade_2025[coluna_indice] = (
        ranking_intensidade_2025["score_intensidade_migratoria"] * peso_migracao
        + ranking_intensidade_2025["score_participacao_emprego"] * peso_emprego
    ) * 100

    ranking_intensidade_2025[coluna_rank] = (
        ranking_intensidade_2025[coluna_indice]
        .rank(ascending=False, method="min")
        .astype(int)
    )

# Compara a estabilidade entre os cenários extremos
correlacao_intensidade = ranking_intensidade_2025[
    ["rank_intensidade_migracao", "rank_intensidade_emprego"]
].corr(method="spearman").iloc[0, 1]

# Identifica municípios comuns aos três Top 20
tops_intensidade = [
    set(ranking_intensidade_2025.nsmallest(20, f"rank_intensidade_{nome}")["codigo_ibge"])
    for nome in pesos_intensidade
]

print(f"Correlação entre cenários: {correlacao_intensidade:.3f}")
print("Municípios comuns aos três Top 20:",
      len(set.intersection(*tops_intensidade)))

Correlação entre cenários: 0.983
Municípios comuns aos três Top 20: 16


## Salvando os rankings

In [66]:
# Recupera a população municipal da base integrada
ranking_intensidade_2025 = ranking_intensidade_2025.merge(
    indicadores_completos[["codigo_ibge", "populacao_2026"]],
    on="codigo_ibge",
    how="left",
    validate="one_to_one"
)

# Confere se os 760 municípios possuem população
print("Municípios:", len(ranking_intensidade_2025))
print("Populações ausentes:",
      ranking_intensidade_2025["populacao_2026"].isna().sum())

Municípios: 760
Populações ausentes: 0


In [67]:
from pathlib import Path

# Define a pasta de saída dos rankings
pasta_saida = Path("../data/processed/rankings")
pasta_saida.mkdir(parents=True, exist_ok=True)

# Seleciona as colunas relevantes para análise e visualização
colunas_ranking = [
    "codigo_ibge", "municipio", "uf",
    "2025", "admissoes", "admissoes_gerais",
    "populacao_2026",
    "registros_por_10mil_hab",
    "participacao_migrante_pct",
    "indice_oportunidade", "rank_equilibrado",
    "indice_intensidade", "rank_intensidade",
    "rank_prioridade_migracao", "rank_prioridade_emprego",
    "rank_intensidade_migracao", "rank_intensidade_emprego"
]

# Organiza os resultados e salva o ranking comparativo
ranking_final_2025 = ranking_intensidade_2025[
    colunas_ranking
].sort_values("rank_equilibrado")

ranking_final_2025.to_parquet(
    pasta_saida / "ranking_municipal_volume_intensidade_2025.parquet",
    index=False
)

# Verifica o arquivo consolidado
print("Municípios:", len(ranking_final_2025))
print("Códigos duplicados:",
      ranking_final_2025["codigo_ibge"].duplicated().sum())

display(ranking_final_2025.head())

Municípios: 760
Códigos duplicados: 0


,codigo_ibge,municipio,uf,2025,admissoes,admissoes_gerais,populacao_2026,registros_por_10mil_hab,participacao_migrante_pct,indice_oportunidade,rank_equilibrado,indice_intensidade,rank_intensidade,rank_prioridade_migracao,rank_prioridade_emprego,rank_intensidade_migracao,rank_intensidade_emprego
0,3550308.0,SÃO PAULO,SP,38523.0,34973.0,2894950.0,11911337,32.341458,1.208069,100.00,1,35.623765,304,1,1,282,332
1,4106902.0,CURITIBA,PR,12861.0,34731.0,570786.0,1832183,70.194953,6.084767,94.77,2,55.692333,100,2,2,95,122
2,1400100.0,BOA VISTA,RR,35014.0,7464.0,44363.0,500965,698.931063,16.824832,92.17,3,83.961822,4,3,3,3,4
3,4204202.0,CHAPECÓ,SC,4399.0,12807.0,69481.0,289170,152.125048,18.432377,84.93,4,73.864322,11,5,4,12,12
4,1302603.0,MANAUS,AM,8759.0,5587.0,279936.0,2327101,37.639105,1.995813,84.22,5,40.500332,263,4,8,245,279
